# Collecte et exploration : commentaires YouTube
Ce notebook utilise le même code que `run.py` : collecte, validation, puis exploration du corpus.

In [ ]:
%pip install -r ../requirements.txt

## 1. Initialisation

In [ ]:
import sys, json, glob
from pathlib import Path

# Racine du projet (le notebook est dans notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import run                      # contient le pipeline complet
from src import collector, validation

# Recharge automatiquement src/*.py si on les modifie
%load_ext autoreload
%autoreload 2
print("Racine du projet :", ROOT)

## 2. Tester la clé API
Doit afficher `Code HTTP : 200`.

In [ ]:
run.main(["--check-key"])

## 3. Lancer la collecte
Modifier les paramètres puis exécuter. Commencer petit (3 vidéos) pour économiser le quota (`search.list` coûte 100 unités sur 10 000 par jour).

In [ ]:
args = [
    "--query", '"artificial intelligence"|"machine learning"|"ChatGPT"',
    "--max-videos", "8",
    "--max-comments", "300",
    "--min-video-comments", "100",
    "--comment-order", "relevance",
    "--published-after", "2025-01-01",
    "--published-before", "2026-10-27",
    "--filter-langs", "en,fr",
    "--relevance-lang", "en",
]
code_retour = run.main(args)

## 4. Charger le corpus final

In [ ]:
files = sorted(glob.glob(str(ROOT / "data" / "analytical" / "corpus_*.csv")))
print("Fichier :", files[-1])
df = pd.read_csv(files[-1])
df.head()

## 5. Vue d'ensemble

In [ ]:
print("Nombre de commentaires :", len(df))
print("Nombre de vidéos        :", df["video_id"].nunique())
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)
print("Période                 :", df["published_at"].min(), "->", df["published_at"].max())
df["n_chars"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().str.len()
df[["n_chars", "n_words", "like_count"]].describe()

## 6. Répartitions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df["lang"].value_counts().head(8).plot.bar(ax=axes[0], title="Langues")
df["video_title"].str[:30].value_counts().plot.barh(ax=axes[1], title="Commentaires par vidéo")
df["n_words"].plot.hist(ax=axes[2], bins=30, title="Longueur (mots)")

plt.tight_layout()
plt.show()

## 7. Commentaires les plus aimés

In [ ]:
df.sort_values("like_count", ascending=False)[["text", "like_count", "lang"]].head(10)

## 8. Métadonnées et rapport de validation

In [ ]:
meta_files = sorted(glob.glob(str(ROOT / "data" / "metadata" / "collection_info_*.json")))
with open(meta_files[-1], encoding="utf-8") as f:
    meta = json.load(f)

print("Requête :", meta["parametres"]["requete"])
print("Période de collecte :", meta["date_collecte_debut"], "->", meta["date_collecte_fin"])
print("Quota consommé (estimé) :", meta["statistiques_collecte"]["quota_units"], "unités")
meta["validation"]